
## EDA SILVER - Calidad y distribucion de los datos limpios
## (pf.silver.modelos + pf.silver.model_tag)


In [0]:
%sql

SELECT
    COUNT(*) AS filas,
    COUNT(DISTINCT model_id) AS modelos_unicos,
    COUNT(DISTINCT ingestion_date) AS dietas
FROM pf.silver.modelos;


In [0]:
%sql
SELECT
  id_y_dia,
  cnt
FROM
  (
    SELECT
      model_id || '|' || CAST(ingestion_date AS STRING) AS id_y_dia,
      COUNT(*) AS cnt
    FROM
      pf.silver.modelos
    GROUP BY
      model_id,
      ingestion_date
  )
WHERE
  cnt > 1
LIMIT 5;

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  SUM(
    CASE
      WHEN has_metadata THEN 1
      ELSE 0
    END
  ) AS con_payload,
  SUM(
    CASE
      WHEN pipeline_tag IS NULL THEN 1
      ELSE 0
    END
  ) AS sin_task,
  SUM(
    CASE
      WHEN library_name IS NULL THEN 1
      ELSE 0
    END
  ) AS sin_libreria,
  SUM(
    CASE
      WHEN license_tag IS NULL THEN 1
      ELSE 0
    END
  ) AS sin_licencia,
  SUM(
    CASE
      WHEN org_id IS NULL THEN 1
      ELSE 0
    END
  ) AS sin_org,
  SUM(
    CASE
      WHEN createdAt IS NULL THEN 1
      ELSE 0
    END
  ) AS sin_created
FROM
  pf.silver.modelos;

In [0]:
%sql
SELECT
  ingestion_date,
  MIN(createdAt) AS created_min,
  MAX(createdAt) AS created_max,
  COUNT(*) AS filas
FROM
  pf.silver.modelos
GROUP BY
  ingestion_date
ORDER BY
  ingestion_date;

In [0]:
%sql
SELECT
  PERCENTILE(downloads, 0.25) AS p25,
  PERCENTILE(downloads, 0.5) AS mediana,
  PERCENTILE(downloads, 0.90) AS p90,
  PERCENTILE(downloads, 0.99) AS p99,
  MAX(downloads) AS max,
  STDDEV(downloads) AS desvio
FROM
  pf.silver.modelos
WHERE
  downloads IS NOT NULL;

In [0]:
%sql
SELECT
  CASE
    WHEN downloads = 0 THEN 'sin_descargas'
    WHEN downloads < 100 THEN '1-99'
    WHEN downloads < 1000 THEN '100-999'
    WHEN downloads < 10000 THEN '1k-10k'
    WHEN downloads < 100000 THEN '10k-100k'
    ELSE '>100k'
  END AS bucket,
  COUNT(DISTINCT model_id) AS modelos
FROM
  pf.silver.modelos
WHERE
  ingestion_date
    = (
      SELECT
        MAX(ingestion_date)
      FROM
        pf.silver.modelos
    )
GROUP BY
  1
ORDER BY
  2 DESC;

In [0]:
%sql
SELECT
  model_id,
  nombre,
  org_id,
  likes,
  downloads,
  pipeline_tag,
  license_tag
FROM
  pf.silver.modelos
WHERE
  ingestion_date
    = (
      SELECT
        MAX(ingestion_date)
      FROM
        pf.silver.modelos
    )
ORDER BY
  likes DESC
LIMIT 15;

In [0]:
%sql
SELECT
  org_id,
  COUNT(DISTINCT model_id) AS modelos,
  SUM(downloads) AS descargas
FROM
  pf.silver.modelos
WHERE
  ingestion_date
    = (
      SELECT
        MAX(ingestion_date)
      FROM
        pf.silver.modelos
    )
GROUP BY
  org_id
ORDER BY
  modelos DESC
LIMIT 15;

In [0]:
%sql
SELECT
  COALESCE(license_tag, '(sin licencia)') AS license_tag,
  COUNT(DISTINCT model_id) AS modelos,
  SUM(downloads) AS descargas
FROM
  pf.silver.modelos
WHERE
  ingestion_date
    = (
      SELECT
        MAX(ingestion_date)
      FROM
        pf.silver.modelos
    )
GROUP BY
  license_tag
ORDER BY
  modelos DESC
LIMIT 15;

In [0]:
%sql
SELECT
  ingestion_date,
  COUNT(*) AS tags_total,
  SUM(
    CASE
      WHEN es_license THEN 1
      ELSE 0
    END
  ) AS tags_license,
  SUM(
    CASE
      WHEN es_dataset THEN 1
      ELSE 0
    END
  ) AS tags_dataset,
  SUM(
    CASE
      WHEN es_arxiv THEN 1
      ELSE 0
    END
  ) AS tags_arxiv
FROM
  pf.silver.model_tag
GROUP BY
  ingestion_date
ORDER BY
  ingestion_date;